In [17]:
import pandas as pd

df = pd.read_csv("games_dataset.csv")
df_regions = pd.read_csv("team_region_groups.csv")

print("Games Dataset:")
print(df.head())
print("\nTeam Region Groups:")
print(df_regions.head())


Games Dataset:
          game_id   game_date                       team  FGA_2  FGM_2  FGA_3  \
0  game_2022_2011  2021-12-30      georgia_lady_bulldogs     50     22     11   
1  game_2022_2011  2021-12-30                 lsu_tigers     50     24     11   
2  game_2022_2012  2021-12-30            missouri_tigers     43     18     15   
3  game_2022_2012  2021-12-30   south_carolina_gamecocks     55     23     21   
4  game_2022_2013  2021-12-30  tennessee_lady_volunteers     41     20     15   

   FGM_3  FTA  FTM  AST  ...  largest_lead  notD1_incomplete  \
0      5    6    3   14  ...           1.0             False   
1      4   15    8   15  ...          14.0             False   
2      7   16   13   10  ...           8.0             False   
3      6    9    5   15  ...           6.0             False   
4      4   15   10   16  ...          19.0             False   

   OT_length_min_tot  rest_days  attendance  tz_dif_H_E  prev_game_dist  \
0                NaN        9.0      3

In [23]:
# claculating wins and losses using team_score and opponent_team_score,, win: 1(team_score>opponent_team_score; 0 for loss)
df['result'] = df.apply(lambda row: 1 if row['team_score'] > row['opponent_team_score'] else 0, axis=1)
print(df['result'].head(16))

0     0
1     1
2     1
3     0
4     1
5     0
6     0
7     1
8     1
9     0
10    0
11    1
12    0
13    1
14    0
15    1
Name: result, dtype: int64


In [25]:
# Aggregate performance metrics
team_performance = df.groupby('team').agg(
    wins=('result', 'sum'),
    losses=('result', lambda x: len(x) - x.sum()),
    total_points=('team_score', 'sum'),
    total_games=('game_id', 'count'),
    avg_points=('team_score', 'mean'),
    avg_opponent_score=('opponent_team_score', 'mean'),
    avg_FGA_2=('FGA_2', 'mean'),
    avg_FGM_2=('FGM_2', 'mean'),
    avg_FGA_3=('FGA_3', 'mean'),
    avg_FGM_3=('FGM_3', 'mean'),
    avg_FTA=('FTA', 'mean'),
    avg_FTM=('FTM', 'mean'),
    avg_AST=('AST', 'mean'),
    avg_BLK=('BLK', 'mean'),
    avg_STL=('STL', 'mean'),
    avg_TOV=('TOV', 'mean'),
    avg_DREB=('DREB', 'mean'),
    avg_OREB=('OREB', 'mean'),
    avg_F_tech=('F_tech', 'mean'),
    avg_F_personal=('F_personal', 'mean'),
).reset_index()


In [53]:
# team and the region it's found in
team_rankings = pd.merge(team_performance, df_regions, on='team', how='inner')
# Filter for the desired regions(West, North, South  onyl)
filtered_rankings = team_rankings[team_rankings['region'].isin(['West', 'North', 'South'])]

filtered_rankings['ranking_score'] = (
    filtered_rankings['wins'] + 
    0.5 * (filtered_rankings['avg_points'] - filtered_rankings['avg_opponent_score'])
)
# sorting using ranking_score
ranked_teams = filtered_rankings.sort_values(by='ranking_score', ascending=False)
# only top-16
top_teams = ranked_teams.groupby('region').head(16)

top_teams.shape
print(top_teams)

                               team  wins  losses  total_points  total_games  \
99         south_carolina_gamecocks    29       2          2210           31   
28        florida_gulf_coast_eagles    27       2          2258           29   
107               stanford_cardinal    28       3          2287           31   
10                     baylor_bears    27       6          2541           33   
108      stephen_f_austin_ladyjacks    25       4          2160           29   
12                      byu_cougars    24       3          2138           27   
115                 texas_longhorns    26       6          2324           32   
61             louisville_cardinals    25       4          2093           29   
101            south_dakota_coyotes    25       5          2094           30   
36                 gonzaga_bulldogs    26       6          2218           32   
47              iowa_state_cyclones    26       6          2460           32   
117                  toledo_rockets    2

In [55]:
print(top_teams[['team', 'region', 'wins', 'losses', 'avg_points', 'avg_opponent_score', 'ranking_score']])

                               team region  wins  losses  avg_points  \
99         south_carolina_gamecocks  North    29       2   71.290323   
28        florida_gulf_coast_eagles  North    27       2   77.862069   
107               stanford_cardinal   West    28       3   73.774194   
10                     baylor_bears   West    27       6   77.000000   
108      stephen_f_austin_ladyjacks  North    25       4   74.482759   
12                      byu_cougars   West    24       3   79.185185   
115                 texas_longhorns   West    26       6   72.625000   
61             louisville_cardinals  South    25       4   72.172414   
101            south_dakota_coyotes   West    25       5   69.800000   
36                 gonzaga_bulldogs   West    26       6   69.312500   
47              iowa_state_cyclones  South    26       6   76.875000   
117                  toledo_rockets  South    26       4   72.366667   
128                unlv_lady_rebels   West    26       6   75.56

In [59]:
top_teams.to_csv('ranked_teams.csv', index=False)
print("Ranked teams saved in .csv file")

Ranked teams saved in .csv file
